<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

Content teams with large portfolios need a short list of pages to review for refresh, instead of guessing. This work uses the FlyRank Hugging Face warehouse: 79,576 pages, 26 clients. Features are Jan–Feb 2026 search signals. The label is a true next-month drop (Apr impressions < 80% of Mar). A fair hand-written rule reaches Precision@50 of 0.640 on client-holdout fold 1. A Random Forest on the same five features reaches 0.280 on that fold (one held-out client; five-fold mean 0.544). The output is a ranked queue with reason codes. It is decision-support for who to look at first, not a claim that a rewrite will move rank.

## 1. Question

**Research Question:** Can a short ranked list, built from 90-day search signals, help editors decide which pages to review for refresh first?

**Decision this supports:** A queue with action labels and reason codes. Editors still decide; the model only orders the pile.

**Unit of analysis:** One content page (`content_id`)

**Output:** Ranked queue. Rules define four labels; on this warehouse run only CONTENT_REFRESH_PRIORITY and MONITOR_STABLE appeared.

This notebook summarizes the Refresh / Content Opportunity Scoring track.

**Notebooks:** w01 research question, w02 task framing, w03 data contract, w04 baseline, w05 model, w06 validation, w07 playbook.

**Files:** `work/capstone_report.md`, `work/presentation.md`, `work/outputs/canonical_metrics.json`, `docs/paper/`.

## 2. Data

**Source:** FlyRank Hugging Face warehouse `hf://datasets/FlyRank/internship-warehouse` (build v20260703)

**Size:** 79,576 content pages after filters (26 clients)

**Grain:** One row per `client_hash_id` × `content_hash_id`

**Feature window:** Jan–Feb 2026 (before the decision date)

**Label window:** Mar vs Apr 2026. Down if Apr impressions < 80% of Mar. This is a future outcome, not the starter-CSV same-window proxy.

**Key features used:**
- `content_age_days`: Age at 2026-03-01 from `content_created_date`
- `days_since_last_update`: Freshness from `content_updated_date`
- `impressions_90d`: Jan–Feb impressions
- `ctr`: clicks / impressions in that window (stored ×100, same as the CSV)
- `avg_position`: impression-weighted GSC position

**Exclusions:**
- `trend_pct` / the Apr–Mar ratio: label-derived, left out
- Pages with < 50 feature-window impressions or < 50 Mar impressions
- Clients whose GSC history starts after 2026-01-01

**Public-safe:** IDs are hashes. No private URLs, client names, or raw queries.

In [5]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    scripts = p / "work" / "scripts"
    if (scripts / "warehouse_frame.py").exists():
        sys.path.insert(0, str(scripts))
        break
else:
    raise FileNotFoundError("work/scripts/warehouse_frame.py not found")

from warehouse_frame import load_notebook_frame

df = load_notebook_frame()
print(f"Label distribution:\n{df['trend_direction'].value_counts()}")

Hugging Face warehouse: 79,576 pages, 26 clients
Features: Jan-Feb 2026. Label: Apr impressions < 80% of Mar.
Declining rate: 0.557
Label distribution:
trend_direction
down      44314
stable    19734
up        15528
Name: count, dtype: int64


In [6]:
import json
import os

metrics_paths = [
    "work/outputs/canonical_metrics.json",
    "../outputs/canonical_metrics.json",
    "outputs/canonical_metrics.json",
]
metrics = None
for path in metrics_paths:
    if os.path.exists(path):
        with open(path) as f:
            metrics = json.load(f)
        print(path)
        break

if metrics is None:
    raise FileNotFoundError("canonical_metrics.json not found")

print(f"base_rate {metrics['base_rate']:.3f}")
print(f"fair_baseline_precision_at_50 {metrics['fair_baseline_precision_at_50']:.3f}")
print(f"random_forest_precision_at_50 {metrics['random_forest_precision_at_50']:.3f}")
print(f"model_vs_baseline_ratio {metrics['model_vs_baseline_ratio']:.2f}")
print(f"model_vs_random_ratio {metrics['model_vs_random_ratio']:.2f}")
print("feature_importance")
for name, value in metrics["feature_importance"].items():
    print(f"  {name} {value:.3f}")

../outputs/canonical_metrics.json
base_rate 0.439
fair_baseline_precision_at_50 0.640
random_forest_precision_at_50 0.280
model_vs_baseline_ratio 0.44
model_vs_random_ratio 0.64
feature_importance
  content_age_days 0.246
  days_since_last_update 0.422
  impressions_90d 0.103
  ctr 0.147
  avg_position 0.082


## 3. Methodology

**Assumptions:**
- Jan–Feb 2026 is knowable at the 2026-03-01 decision
- Apr vs Mar impression drop is a proxy for "worth a refresh look," not proof a rewrite will help
- Client mix matters, so splits are grouped by `client_id`

**Label:** Positive class = Apr impressions < 80% of Mar impressions.

**Features (5):** `content_age_days`, `days_since_last_update`, `impressions_90d`, `ctr`, `avg_position`. Left out: `trend_pct` and the Apr/Mar ratio.

**Baseline:** Fair rule from `w04_baseline_score.ipynb` — stale visible, position > 10 with age, low CTR with high impressions. No label in the score.

**Model:** Random Forest (100 trees, max_depth=8, random_state=42)

**Validation:** GroupKFold, 5 folds. Reported Precision@50 is fold 1 (58,783 / 20,793, one held-out client). Five-fold mean is also reported because fold 1 is a single large client.

**Leakage:** `trend_pct` (Apr/Mar ratio) was tested and left out. Features do not overlap the Apr label month.

## 4. Results (same fold as the report)

Headline metric is Precision@50, not accuracy. Data: Hugging Face warehouse.

| Method | Precision@50 |
|--------|---------------|
| Fair hand-written baseline | 0.640 |
| Random (test-fold base rate) | 0.439 |
| Random Forest (fold 1) | 0.280 |
| Random Forest (5-fold mean) | 0.544 |

On fold 1 the forest does **not** beat the fair rule. That fold is one client (20,793 pages). Five-fold mean is 0.544, near the overall declining rate of 0.557. Importances: days since update, then age, then CTR. Impressions and position are smaller than they were on the starter CSV.

Precision@50 on Hugging Face warehouse, client-holdout fold 1: fair rule 0.640, forest 0.280, test-fold base rate 0.439. Five-fold mean forest 0.544.

Feature importance (warehouse): days_since_last_update 0.422, content_age_days 0.246, ctr 0.147, impressions_90d 0.103, avg_position 0.082.

The ranked queue is built in `w07_action_playbook.ipynb` from out-of-fold model scores. Action labels do not use `trend_direction`.

## 5. Limitations

- The label is Apr vs Mar, after the Jan–Feb features, so this is a future drop, not the starter-CSV same-window proxy.
- Fold 1 is one client (20,793 pages). Five-fold mean (0.544) is the more stable forest number.
- On this warehouse label the forest does not beat the fair rule.
- No experiment, so no claim that a rewrite will recover traffic.
- 26 clients after filters; a new client mix can look different.
- Five features only. No query text.
- `avg_position == 0` is missing rank, not rank zero.
- A person still has to look before anything is published.

## 6. Ranked recommendations

1. Start with CONTENT_REFRESH_PRIORITY: high out-of-fold model score and visible in search.
2. Then STALE_VISIBLE: old pages that still get impressions. On this warehouse run that bucket was empty.
3. Skip REVIEW_THRESHOLD until visibility is fixed (`impressions_90d < 50`). On this warehouse run that bucket was empty.
4. Leave MONITOR_STABLE unless a human has another reason.
5. Before editing: seasonality, brand/legal pages, crawl errors, whether the copy is still true.
6. Track what happened after 30–90 days. The model does not measure that.

Playbook cohorts are computed in `w07_action_playbook.ipynb`. Priority uses model score and visibility, not the label.

## 7. Artifacts the paper embeds

**Charts and tables for the deployed page:**

Precision@50: fair rule 0.640, forest fold-1 0.280, five-fold mean 0.544. Charts for the paper live under `docs/paper/charts/`.

The reported metric is Precision@50, not a 0.5-threshold precision-recall curve.

Top features on the warehouse: days_since_last_update 0.422, content_age_days 0.246, ctr 0.147.

Distribution plots for the paper are generated separately.

Hugging Face warehouse: 79,576 pages, 26 clients. GroupKFold fold 1: train 58,783 / test 20,793 (one client). Five features, random_state=42. Precision@50 fair rule 0.640 vs forest 0.280 vs test base rate 0.439. Five-fold mean forest 0.544.

## ML-12: Demo, Social Post, and Employer Summary

### 5-Minute Demo Outline

**0:00–0:30 — Problem**
- 79,576 warehouse pages, limited editorial time
- Random picking equals the declining rate; that is the base rate, not a win

**0:30–1:30 — Setup**
- Hugging Face warehouse, Jan–Feb features, Apr vs Mar label
- Five features, no `trend_pct`
- Fair rule first (stale / position / low CTR)
- GroupKFold on client_id

**1:30–2:30 — Numbers**
- Fold 1 Precision@50: fair rule 0.640, forest 0.280, test base rate 0.439
- Five-fold mean forest 0.544 (near the 0.557 declining rate)
- The forest does not beat the rule on this future label

**2:30–3:30 — What it leaned on**
- Days since update, then age, then CTR
- Impressions and position were weaker than on the starter CSV

**3:30–4:30 — Queue**
- Rank by model score
- Reason codes without looking up the label
- Human checks before publish

**4:30–5:00 — Limits**
- Future label is harder than the CSV proxy; one-client fold 1; no causal claim

### Social Post Cut

1/4 I ranked 79,576 pages from the FlyRank Hugging Face warehouse for refresh review. Features are Jan–Feb 2026. The label is a next-month drop (Apr vs Mar).

2/4 A fair hand-written rule hits 0.640 Precision@50 on a client-holdout fold. A Random Forest on five signals hits 0.280 on that fold (one client) and 0.544 across five folds — near the 0.557 declining rate.

3/4 That is a negative result: the forest does not beat the rule when the label is in the future. The starter CSV same-window proxy had made the forest look stronger.

4/4 Output is still a ranked queue with reason codes. Editors still decide. No claim that a rewrite will move Google.

### 3-Sentence Employer-Facing Summary

I ranked 79,576 anonymized pages from the FlyRank Hugging Face warehouse for refresh review, using Jan–Feb search signals and an Apr-vs-Mar drop label. On client-holdout fold 1 a fair hand-written rule reaches Precision@50 of 0.640; a Random Forest reaches 0.280 (five-fold mean 0.544). The deliverable is a queue with reason codes for editors, not a claim that an update will recover traffic.

## Reproducibility

**Notebook:** This notebook is available at `work/notebooks/capstone.ipynb`

**Data:** Hugging Face warehouse `hf://datasets/FlyRank/internship-warehouse` via `work/scripts/warehouse_frame.py` (cached to `work/outputs/warehouse_page_frame.parquet`)

**Random seed:** 42 (used for train/test split and model initialization)

**Dependencies:** pandas, numpy, scikit-learn, matplotlib, seaborn

**Outputs:** All generated charts and tables saved to `work/outputs/`

## Acknowledgments & Data Credit

This work uses data provided by [FlyRank](https://flyrank.ai) as part of their internship program. The dataset includes anonymized content performance metrics from multiple clients and is used here for educational and research purposes. FlyRank's internship program provides real-world datasets for machine learning education, enabling students to work with meaningful data while protecting client privacy.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
